# Figure 2: Repertoire size and TCR diversity as a function of age and sex

This notebook reproduces Figure 2 of the manuscript.

The figure shows:

- repertoire size, \(S\), as a function of age and sex,
- TCR diversity, \(D\), as a function of age and sex,
- the same quantities after shifting female samples to younger ages by the best-fit age offset.

The age offset is estimated by finding the horizontal shift of the female curves that best aligns them with the male curves. The manuscript reports an offset of \(11.4 \pm 0.9\) years.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline
from scipy.optimize import brute

# Render figures cleanly in notebooks.
%config InlineBackend.figure_format = "pdf"

# Global plotting defaults.
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["axes.labelsize"] = 16

# Use Matplotlib default color cycle.
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Set a random seed so bootstrap calculations are reproducible.
rng = np.random.default_rng(1)

In [2]:
# Public data table hosted on Zenodo.
# This table contains subject-level TCR repertoire metrics used in the manuscript.
data_url = "https://zenodo.org/records/13993996/files/TCR_size_diversity_age_sex.csv?download=1"

df = (
    pd.read_csv(data_url)
    .rename(
        columns={
            "#Subject_ID": "subject_id",
            " age": "age",
            " sex": "sex",
            " size": "repertoire_size",
            " diversity": "tcr_diversity",
        }
    )
)

# Remove leading/trailing whitespace from sex labels.
df["sex"] = df["sex"].str.strip().str.lower()

# Add log-transformed quantities used for Figure 2.
df["log_repertoire_size"] = np.log10(df["repertoire_size"])
df["log_tcr_diversity"] = np.log10(df["tcr_diversity"])

df.head()

,subject_id,age,sex,repertoire_size,tcr_diversity,log_repertoire_size,log_tcr_diversity
0,30000,63,male,508139,274815,5.705983,5.439040
1,30001,76,female,263716,167611,5.421136,5.224303
2,30002,70,female,519416,261836,5.715515,5.418029
3,30003,44,male,576294,383460,5.760644,5.583720
4,30004,47,male,505447,290642,5.703676,5.463358


In [3]:
def bootstrap_statistic(values, statistic=np.median, n_bootstrap=1001, rng=None):
    """
    Estimate the bootstrap uncertainty of a statistic.

    Parameters
    ----------
    values : array-like
        Values to bootstrap.
    statistic : callable
        Statistic to calculate. Default is np.median.
    n_bootstrap : int
        Number of bootstrap realizations.
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    point_estimate : float
        Statistic evaluated on the original data.
    bootstrap_error : float
        Standard deviation of bootstrap statistics.
    """
    if rng is None:
        rng = np.random.default_rng()

    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    bootstrap_values = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_values.append(statistic(sample))

    return statistic(values), np.std(bootstrap_values)


def binned_summary(x, y, bins, interval=(0.25, 0.75), n_bootstrap=1001, rng=None):
    """
    Calculate binned medians, bootstrap errors on the median,
    and central intervals of y in bins of x.

    Parameters
    ----------
    x : array-like
        Binning variable, usually subject age.
    y : array-like
        Quantity to summarize.
    bins : list of tuple
        List of (lower, upper) bin edges.
    interval : tuple
        Lower and upper quantiles for the central interval.
        For Figure 2, we use (0.25, 0.75), the central 50%.
    n_bootstrap : int
        Number of bootstrap realizations for median uncertainty.
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    summary : pd.DataFrame
        One row per age bin with median x, median y, bootstrap error,
        and central interval bounds.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.asarray(x)
    y = np.asarray(y)

    rows = []

    for lower, upper in bins:
        in_bin = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x >= lower)
            & (x < upper)
        )

        x_bin = x[in_bin]
        y_bin = y[in_bin]

        if len(y_bin) <= 1:
            continue

        y_median, y_error = bootstrap_statistic(
            y_bin,
            statistic=np.median,
            n_bootstrap=n_bootstrap,
            rng=rng,
        )

        rows.append(
            {
                "age_median": np.median(x_bin),
                "value_median": y_median,
                "value_error": y_error,
                "interval_low": np.quantile(y_bin, interval[0]),
                "interval_high": np.quantile(y_bin, interval[1]),
                "n_subjects": len(y_bin),
            }
        )

    return pd.DataFrame(rows)

In [4]:
# Decade-wide bins from 0--10 through 80--100.
# The final bin is extended to 100 to include the oldest subjects.
age_bin_lower = np.arange(9) * 10
age_bin_upper = age_bin_lower + 10
age_bin_upper[-1] = 100

age_bins = list(zip(age_bin_lower, age_bin_upper))

male = df["sex"] == "male"
female = df["sex"] == "female"

print(f"N male:   {male.sum():,}")
print(f"N female: {female.sum():,}")
print(f"Female fraction: {female.mean():.3f}")

N male:   14,371
N female: 15,984
Female fraction: 0.525


In [5]:
# Figure 2 shows the central 50% of subjects in each age bin.
central_50 = (0.25, 0.75)

# Number of bootstrap realizations.
# The manuscript used 10,001 for final quoted values.
# 1,001 is faster and visually indistinguishable for plotting.
n_bootstrap = 1001

male_size = binned_summary(
    df.loc[male, "age"],
    df.loc[male, "log_repertoire_size"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

female_size = binned_summary(
    df.loc[female, "age"],
    df.loc[female, "log_repertoire_size"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

male_diversity = binned_summary(
    df.loc[male, "age"],
    df.loc[male, "log_tcr_diversity"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

female_diversity = binned_summary(
    df.loc[female, "age"],
    df.loc[female, "log_tcr_diversity"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

male_diversity

,age_median,value_median,value_error,interval_low,interval_high,n_subjects
0,18.0,5.617397,0.012310,5.513252,5.714669,272
1,25.0,5.575863,0.007302,5.484062,5.667769,754
2,36.0,5.542040,0.002795,5.450043,5.629141,2564
3,45.0,5.507006,0.002735,5.405862,5.596707,3642
4,54.0,5.460493,0.003137,5.357161,5.560541,3777
5,63.0,5.408192,0.004437,5.290957,5.511951,2475
6,73.0,5.328278,0.007101,5.199395,5.453289,779
7,82.0,5.169589,0.029870,5.019467,5.349419,108


In [6]:
# ============================================================
# Estimate female age offset using the original manuscript procedure
# ============================================================
#
# This cell follows the implementation in the original manuscript notebook:
# - bootstrap male and female repertoires independently
# - compute binned medians for log TCR diversity and log repertoire size
# - shift the female curves younger by delta_age
# - minimize the joint squared residuals relative to male curves
#
# The manuscript value was computed using 10,001 bootstrap realizations:
#     11.4 ± 0.9 years
#
# For a quick test, set n_offset_bootstrap = 101.
# For manuscript reproduction, use n_offset_bootstrap = 10001.

from scipy.interpolate import CubicSpline
from scipy.optimize import brute

np.random.seed(1)

# Original notebook used these decade-wide bins.
b1 = np.arange(9) * 10
b2 = np.arange(9) * 10 + 10
b2[-1] = 100
bins = [(lo, hi) for lo, hi in zip(b1, b2)]

# Original notebook-style binned median function.
# This is included here so the offset calculation does not depend on
# the refactored binned_summary function used elsewhere in this notebook.
def bootbin_median_original(
    _x,
    _y,
    nbins=20,
    nreal=1001,
    ci=[0.1, 0.9],
    bins=None,
    do_mean=False,
    do_sum=False,
):
    good_index = np.where(np.isfinite(_x) & np.isfinite(_y))[0]
    x = _x[good_index]
    y = _y[good_index]

    ss = np.argsort(x)
    xss = x[ss]
    yss = y[ss]

    if bins is None:
        x_split = np.array_split(xss, nbins)
        y_split = np.array_split(yss, nbins)
    else:
        x_split = []
        y_split = []
        for bi in bins:
            iii = np.where((xss >= bi[0]) & (xss < bi[1]))[0]
            if len(iii) > 1:
                x_split.append(xss[iii])
                y_split.append(yss[iii])

    if do_mean:
        agg_func_x = np.mean
        agg_func_y = np.mean
    elif do_sum:
        agg_func_x = np.median
        agg_func_y = np.sum
    else:
        agg_func_x = np.median
        agg_func_y = np.median

    xmed = np.asarray([agg_func_x(xi) for xi in x_split])
    ymed = np.asarray([agg_func_y(yi) for yi in y_split])

    ci_lo = np.asarray(
        [
            np.interp(ci[0], np.linspace(0, 1, num=len(yi)), np.sort(yi))
            for yi in y_split
        ]
    )
    ci_hi = np.asarray(
        [
            np.interp(ci[1], np.linspace(0, 1, num=len(yi)), np.sort(yi))
            for yi in y_split
        ]
    )

    if not do_sum:
        yboot = [
            [
                agg_func_y(
                    yi[np.random.randint(0, high=len(yi), size=len(yi))]
                )
                for yi in y_split
            ]
            for _ in range(nreal)
        ]
        yerr = np.asarray([np.nanstd(yi) for yi in np.asarray(yboot).T])
    else:
        yerr = np.sqrt(ymed)

    return {
        "xmed": xmed,
        "ymed": ymed,
        "yerr": yerr,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
    }


# Use the refactored public dataframe.
# Expected columns:
#   age
#   sex
#   repertoire_size
#   tcr_diversity
male_index = np.where(df["sex"].values == "male")[0]
female_index = np.where(df["sex"].values == "female")[0]

ages_for_offset_fit = np.arange(18, 70)
offset_search_range = (slice(5, 25, 0.2),)


def age_offset_objective_original(delta_age, make_plot=False):
    """
    Objective function from the original notebook.

    The female curves are shifted younger by delta_age.
    The objective compares both:
    - log TCR diversity
    - log repertoire size
    """
    delta_age = float(np.asarray(delta_age))

    # Female curves shifted younger.
    female_diversity_spline = CubicSpline(
        female_diversity_boot["xmed"] - delta_age,
        female_diversity_boot["ymed"],
    )
    female_size_spline = CubicSpline(
        female_size_boot["xmed"] - delta_age,
        female_size_boot["ymed"],
    )

    # Male curves.
    male_diversity_spline = CubicSpline(
        male_diversity_boot["xmed"],
        male_diversity_boot["ymed"],
    )
    male_size_spline = CubicSpline(
        male_size_boot["xmed"],
        male_size_boot["ymed"],
    )

    objective = np.sum(
        (
            female_diversity_spline(ages_for_offset_fit)
            - male_diversity_spline(ages_for_offset_fit)
        ) ** 2
        +
        (
            female_size_spline(ages_for_offset_fit)
            - male_size_spline(ages_for_offset_fit)
        ) ** 2
    )

    if make_plot:
        plt.plot(
            ages_for_offset_fit,
            female_diversity_spline(ages_for_offset_fit),
            label="Female shifted diversity",
        )
        plt.plot(
            ages_for_offset_fit,
            male_diversity_spline(ages_for_offset_fit),
            label="Male diversity",
        )
        plt.title(f"Objective = {objective:.4g}")
        plt.legend()
        plt.show()

    return objective


# Set to 10001 for manuscript reproduction.
n_offset_bootstrap = 201 #change to large number for more accurate errorbar

offsets = []

for i in range(n_offset_bootstrap):
    male_resample = np.random.choice(
        np.arange(len(male_index)),
        size=len(male_index),
        replace=True,
    )
    female_resample = np.random.choice(
        np.arange(len(female_index)),
        size=len(female_index),
        replace=True,
    )

    male_diversity_boot = bootbin_median_original(
        df["age"].values[male_index][male_resample],
        np.log10(df["tcr_diversity"].values[male_index][male_resample]),
        bins=bins,
        nreal=3,
    )

    female_diversity_boot = bootbin_median_original(
        df["age"].values[female_index][female_resample],
        np.log10(df["tcr_diversity"].values[female_index][female_resample]),
        bins=bins,
        nreal=3,
    )

    male_size_boot = bootbin_median_original(
        df["age"].values[male_index][male_resample],
        np.log10(df["repertoire_size"].values[male_index][male_resample]),
        bins=bins,
        nreal=3,
    )

    female_size_boot = bootbin_median_original(
        df["age"].values[female_index][female_resample],
        np.log10(df["repertoire_size"].values[female_index][female_resample]),
        bins=bins,
        nreal=3,
    )

    try:
        best_offset_this_bootstrap = float(
            brute(
                age_offset_objective_original,
                offset_search_range,
                finish=None,
            )
        )
        offsets.append(best_offset_this_bootstrap)
    except Exception as error:
        print(f"Bootstrap realization {i} failed: {error}")

offsets = np.asarray(offsets)

female_age_offset = np.mean(offsets)
female_age_offset_error = np.std(offsets)

print(
    f"Female age offset = "
    f"{female_age_offset:.1f} ± {female_age_offset_error:.1f} years"
)
print(f"N successful bootstrap realizations = {len(offsets):,}")
print(female_age_offset)

/var/folders/x5/vzmwmhlx37g6_cjs3b49_t640000gn/T/ipykernel_21730/1230315826.py:132: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  delta_age = float(np.asarray(delta_age))


Female age offset = 11.4 ± 0.9 years
N successful bootstrap realizations = 201
11.360199004975131


In [7]:
# ============================================================
# Compute shifted-female summaries for Figure 2C-D
# ============================================================
#
# Use the manuscript age offset for the final plotted figure.
# The previous cell independently reproduces this value as ~11.4 ± 0.9 years
# when run with 10,001 bootstrap realizations.

female_age_shift = 11.4

female_shifted_size = binned_summary(
    df.loc[female, "age"] - female_age_shift,
    df.loc[female, "log_repertoire_size"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

female_shifted_diversity = binned_summary(
    df.loc[female, "age"] - female_age_shift,
    df.loc[female, "log_tcr_diversity"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

female_shifted_summary = pd.DataFrame(
    {
        "shifted_age_median": female_shifted_diversity["age_median"],
        "female_shifted_logS": female_shifted_size["value_median"],
        "female_shifted_logS_error": female_shifted_size["value_error"],
        "female_shifted_logD": female_shifted_diversity["value_median"],
        "female_shifted_logD_error": female_shifted_diversity["value_error"],
        "n_subjects": female_shifted_diversity["n_subjects"],
    }
)

female_shifted_summary

,shifted_age_median,female_shifted_logS,female_shifted_logS_error,female_shifted_logD,female_shifted_logD_error,n_subjects
0,7.6,5.744205,0.006629,5.620819,0.007804,469
1,16.6,5.751882,0.005628,5.595397,0.005966,903
2,25.6,5.737469,0.002861,5.561961,0.003381,2928
3,35.6,5.718693,0.002425,5.523495,0.003045,4157
4,44.6,5.731192,0.001896,5.521698,0.002558,4346
5,53.6,5.706256,0.003164,5.470972,0.004283,2481
6,62.6,5.680413,0.008213,5.414416,0.011094,620
7,72.6,5.635839,0.024443,5.312960,0.027792,80


In [8]:
# ============================================================
# Plotting helper for sex-stratified binned curves
# ============================================================

def plot_two_binned_curves(
    ax,
    summary_a,
    summary_b,
    label_a,
    label_b,
    ylabel,
    panel_label,
    legend_loc="lower left",
):
    """
    Plot two age-binned curves with:
    - central 50% shaded regions
    - bootstrap errors on the median
    - consistent panel labeling

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis on which to draw the panel.
    summary_a, summary_b : pd.DataFrame
        Binned summaries produced by binned_summary().
        Each must contain:
        age_median, value_median, value_error, interval_low, interval_high.
    label_a, label_b : str
        Legend labels for the two groups.
    ylabel : str
        Y-axis label.
    panel_label : str
        Panel label, e.g. "(A)".
    legend_loc : str
        Matplotlib legend location.
    """
    ax.fill_between(
        summary_a["age_median"],
        summary_a["interval_low"],
        summary_a["interval_high"],
        alpha=0.5,
        color=colors[0],
        label=label_a,
    )

    ax.errorbar(
        summary_a["age_median"],
        summary_a["value_median"],
        yerr=summary_a["value_error"],
        linewidth=3,
        color=colors[0],
    )

    ax.fill_between(
        summary_b["age_median"],
        summary_b["interval_low"],
        summary_b["interval_high"],
        alpha=0.5,
        color=colors[1],
        label=label_b,
    )

    ax.errorbar(
        summary_b["age_median"],
        summary_b["value_median"],
        yerr=summary_b["value_error"],
        linewidth=3,
        color=colors[1],
    )

    ax.set_xlabel("Age")
    ax.set_ylabel(ylabel)

    ax.legend(
        fontsize=12,
        loc=legend_loc,
        frameon=True,
    )

    ax.text(
        0.93,
        0.93,
        panel_label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="right",
        va="top",
    )

In [9]:
# ============================================================
# Plot Figure 2
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

plot_two_binned_curves(
    axes[0, 0],
    male_size,
    female_size,
    label_a="Male, Central 50%",
    label_b="Female, Central 50%",
    ylabel="Repertoire Size (Log $S$)",
    panel_label="(A)",
    legend_loc="lower left",
)

plot_two_binned_curves(
    axes[0, 1],
    male_diversity,
    female_diversity,
    label_a="Male, Central 50%",
    label_b="Female, Central 50%",
    ylabel="Repertoire Diversity (Log $D$)",
    panel_label="(B)",
    legend_loc="lower left",
)

plot_two_binned_curves(
    axes[1, 0],
    male_size,
    female_shifted_size,
    label_a="Male, Central 50%",
    label_b="Female, Central 50%",
    ylabel="Repertoire Size (Log $S$)",
    panel_label="(C)",
    legend_loc="lower left",
)

plot_two_binned_curves(
    axes[1, 1],
    male_diversity,
    female_shifted_diversity,
    label_a="Male, Central 50%",
    label_b="Female, Central 50%",
    ylabel="Repertoire Diversity (Log $D$)",
    panel_label="(D)",
    legend_loc="lower left",
)

fig.tight_layout()

# Uncomment to save the figure.
# fig.savefig("fig2.pdf", bbox_inches="tight")

plt.show()

<Figure size 1500x1200 with 4 Axes>